In [ ]:
import numpy as np
import pickle
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 原始特征提取模块 (简化版)
# ==========================================
def extract_raw_features(segment):
    """
    不再使用 EMD 或 PSD，直接使用原始信号的统计量作为特征
    这里取每个通道在该时间段内的：均值、标准差、最大值、最小值
    """
    all_ch_feats = []
    for ch_data in segment:
        mean_val = np.mean(ch_data)
        std_val = np.std(ch_data)
        max_val = np.max(ch_data)
        min_val = np.min(ch_data)
        all_ch_feats.extend([mean_val, std_val, max_val, min_val])
    return all_ch_feats

# ==========================================
# 2. 实验引擎
# ==========================================
def run_experiment(data_dir):
    subject_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.dat')])
    if not subject_files: raise FileNotFoundError("未找到.dat文件。")
    
    dims = {0: "Valence", 1: "Arousal", 2: "Dominance", 3: "Liking"}
    clfs_pool = {
        "Linear_SVM": svm.SVC(kernel='linear'),
        "RBF_SVM":    svm.SVC(kernel='rbf'),
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
    }
    
    raw_results = []
    # 选取 8 个核心脑电通道 (Fp1, Fp2, F3, F4, C3, C4, P3, P4 等)
    ch_idx = [0, 16, 3, 20, 7, 25, 11, 29]
    
    print(f">>> 开始 32 名受试者定量分析 (原始特征版)...")
    for f_name in subject_files:
        print(f"正在处理: {f_name}       ", end='\r')
        with open(os.path.join(data_dir, f_name), 'rb') as f:
            data_dict = pickle.load(f, encoding='latin1')
        
        # 提取原始数据特征
        X_sub = []
        for i in range(40): # 40个视频
            vid_sig = data_dict['data'][i, ch_idx, :] # 形状: (8, 8064)
            # 将 63 秒信号切分为 12 个 5 秒的分段
            for s in range(0, 8064 - 672 + 1, 672):
                segment = vid_sig[:, s:s+672]
                X_sub.append(extract_raw_features(segment))
        
        X_sub = StandardScaler().fit_transform(np.array(X_sub))
        
        for d_idx, d_name in dims.items():
            y_sub = np.array([1 if data_dict['labels'][i, d_idx] >= 5 else 0 for i in range(40) for _ in range(12)])
            if len(np.unique(y_sub)) < 2: continue 
            
            for c_name, clf in clfs_pool.items():
                y_pred = cross_val_predict(clf, X_sub, y_sub, cv=5)
                raw_results.append({
                    'Subject': f_name, 'Dimension': d_name, 
                    'Classifier': c_name, 'Accuracy': accuracy_score(y_sub, y_pred),
                    'CM': confusion_matrix(y_sub, y_pred)
                })
    return pd.DataFrame(raw_results)

# ==========================================
# 3. 绘图与保存模块 (逻辑同前，修正 Average)
# ==========================================
def save_all_visualizations(df, output_folder="Raw_Experiment_Results"):
    if not os.path.exists(output_folder): os.makedirs(output_folder)
    
    print("\n>>> 正在批量生成 12 组混淆矩阵对比图...")
    for dim in df['Dimension'].unique():
        for clf in df['Classifier'].unique():
            sub_df = df[(df['Dimension'] == dim) & (df['Classifier'] == clf)].sort_values('Accuracy')
            if sub_df.empty: continue
            
            best, worst = sub_df.iloc[-1], sub_df.iloc[0]
            median = sub_df.iloc[len(sub_df)//2]
            avg_cm = np.mean(np.stack(sub_df['CM'].values), axis=0)
            mean_acc_val = sub_df['Accuracy'].mean()

            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            cms = [best['CM'], worst['CM'], median['CM'], avg_cm]
            titles = ["Best Case", "Worst Case", "Median Case", "Average (Overall)"]
            
            for i in range(4):
                sns.heatmap(cms[i], annot=True, fmt='.1f' if i==3 else 'd', cmap='YlOrRd', ax=axes[i])
                if i < 3:
                    idx_map = [0, -1, len(sub_df)//2]
                    current_acc = sub_df.iloc[idx_map[i if i==0 else (1 if i==1 else 2)]]['Accuracy']
                    axes[i].set_title(f"{titles[i]}\nAcc: {current_acc:.2f}")
                else:
                    axes[i].set_title(f"{titles[i]}\nMean Acc: {mean_acc_val:.2f}")
            
            plt.suptitle(f"Raw Feature Analysis: {dim} using {clf}", fontsize=16, fontweight='bold')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.savefig(f"{output_folder}/Raw_CM_{dim}_{clf}.png")
            plt.close()

    # 汇总图
    plt.figure(figsize=(12, 7))
    summary = df.groupby(['Dimension', 'Classifier'])['Accuracy'].mean().unstack()
    summary.plot(kind='bar', edgecolor='black', rot=0, figsize=(12, 7))
    plt.title("Comparison using Raw Statistical Features")
    plt.ylabel("Accuracy")
    plt.savefig(f"{output_folder}/Raw_Overall_Comparison.png")
    plt.show()

# ==========================================
# 4. 主程序
# ==========================================
if __name__ == "__main__":
    INPUT_DIR = "./data_preprocessed_python/" 
    results_df = run_experiment(INPUT_DIR)
    results_df.to_csv("DEAP_Raw_Results.csv", index=False)
    save_all_visualizations(results_df, output_folder="Raw_Experiment_Results")